# Data Preprocessing

This notebook prepares the recruitment dataset for machine learning.

The preprocessing stage includes handling duplicate records, missing values,
categorical variables, and preparing the data for model training while
avoiding data leakage.

## Loading the Dataset

The recruitment dataset is loaded from the raw data directory using Pandas.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/recruitment.csv")

df.head()

,Candidate_ID,Gender,Age,Education_Level,Experience_Years,Skill_Score,Aptitude_Test_Score,Technical_Test_Score,Communication_Score,Certifications_Count,Previous_Companies,Interview_Score,Location,Job_Role_Applied,Expected_Salary,Hiring_Decision
0,1,Male,50,Bachelors,19,48,75,70,65,2.0,5,92,Urban,HR Executive,22214,1
1,2,Other,36,Bachelors,18,9,53,46,25,2.0,4,77,Semi-Urban,Data Analyst,130094,0
2,3,Female,58,Masters,11,1,85,62,72,1.0,3,90,Urban,ML Engineer,78652,0
3,4,Male,48,Masters,0,74,35,79,67,6.0,3,9,Urban,HR Executive,144618,0
4,5,Male,37,Bachelors,0,64,99,25,38,7.0,5,24,Urban,Data Analyst,133865,0


## Handling Duplicate Records

The dataset contains duplicate rows. Duplicate records are removed to prevent
the same candidate record from being represented multiple times during model
training.

In [2]:
duplicate_count = df.duplicated().sum()

duplicate_count

np.int64(1199)

In [3]:
df = df.drop_duplicates().reset_index(drop=True)

df.shape

(120001, 16)

## Handling Missing Values

The dataset contains a small number of missing values in `Gender`,
`Education_Level`, and `Certifications_Count`.

Because `Gender` is the protected attribute used in the fairness analysis,
records with missing gender values are removed rather than assigning an
assumed gender.

In [4]:
df["Gender"].isnull().sum()

np.int64(10)

In [5]:
df = df.dropna(subset=["Gender"]).reset_index(drop=True)

df.shape

(119991, 16)

## Handling Remaining Missing Values

The remaining missing values occur in `Education_Level` and
`Certifications_Count`. Since the number of affected records is very small
relative to the dataset, these records are removed rather than imputed.

In [6]:
df.isnull().sum()

Candidate_ID             0
Gender                   0
Age                      0
Education_Level         10
Experience_Years         0
Skill_Score              0
Aptitude_Test_Score      0
Technical_Test_Score     0
Communication_Score      0
Certifications_Count    10
Previous_Companies       0
Interview_Score          0
Location                 0
Job_Role_Applied         0
Expected_Salary          0
Hiring_Decision          0
dtype: int64

In [7]:
df = df.dropna().reset_index(drop=True)

df.shape

(119971, 16)

## Selecting Variables for the Model

The target variable is `Hiring_Decision`, which represents whether a
candidate was hired or rejected.

`Candidate_ID` is excluded because it is an identifier rather than a
meaningful predictive feature.

`Gender` is retained as a protected attribute for the later fairness
analysis, but is not initially used as a model feature.

The remaining candidate characteristics are considered as potential
predictive features.

In [8]:
target = "Hiring_Decision"

protected_attribute = "Gender"

features = [
    "Age",
    "Education_Level",
    "Experience_Years",
    "Skill_Score",
    "Aptitude_Test_Score",
    "Technical_Test_Score",
    "Communication_Score",
    "Certifications_Count",
    "Previous_Companies",
    "Interview_Score",
    "Location",
    "Job_Role_Applied",
    "Expected_Salary"
]

X = df[features]
y = df[target]

X.head()

,Age,Education_Level,Experience_Years,Skill_Score,Aptitude_Test_Score,Technical_Test_Score,Communication_Score,Certifications_Count,Previous_Companies,Interview_Score,Location,Job_Role_Applied,Expected_Salary
0,50,Bachelors,19,48,75,70,65,2.0,5,92,Urban,HR Executive,22214
1,36,Bachelors,18,9,53,46,25,2.0,4,77,Semi-Urban,Data Analyst,130094
2,58,Masters,11,1,85,62,72,1.0,3,90,Urban,ML Engineer,78652
3,48,Masters,0,74,35,79,67,6.0,3,9,Urban,HR Executive,144618
4,37,Bachelors,0,64,99,25,38,7.0,5,24,Urban,Data Analyst,133865


## Encoding Categorical Variables

Machine learning models require numerical input. The categorical variables
`Education_Level`, `Location`, and `Job_Role_Applied` are therefore converted
into numerical features using one-hot encoding.

A separate column is created for each category, with a value of 1 indicating
that the candidate belongs to that category and 0 otherwise.

## Splitting the Dataset

The dataset is divided into training and testing sets before model training.

80% of the data is used for training and 20% is reserved for testing.
Stratification is used to maintain a similar distribution of the target
variable, `Hiring_Decision`, in both sets.

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((95976, 13), (23995, 13), (95976,), (23995,))

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    "Education_Level",
    "Location",
    "Job_Role_Applied"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [13]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

X_train_processed.shape, X_test_processed.shape

((95976, 23), (23995, 23))

## Checking the Processed Data

The processed training and testing datasets are checked to confirm that the
categorical variables have been successfully converted into numerical
features.

In [14]:
X_train_processed.shape, X_test_processed.shape

((95976, 23), (23995, 23))

## Saving the Cleaned Dataset

The cleaned dataset is saved separately from the raw dataset so that the
original data remains unchanged and the preprocessing steps are reproducible.

In [15]:
df.to_csv("../data/processed/cleaned_recruitment.csv", index=False)

## Conclusion

The dataset has been cleaned and prepared for machine learning. Duplicate
records and missing values were handled, the target and protected attributes
were identified, and categorical variables were encoded after splitting the
data into training and testing sets. The cleaned dataset has also been saved
for reproducibility.